# Phase E.0 — Smoke test for the Akimbo v1.0 production recipe

Validates four things before any of the ~$25 KAN variant runs are launched:

1. **Data**: `linrock/test80-2024` January shard loads through Bullet's `SfBinpackLoader`.
2. **Recipe**: the full Akimbo v1.0 architecture (`(768x4hm -> 1024)x2 -> 16 -> 32 -> 1` with factoriser, 4 input buckets, 8 output buckets) trains end-to-end as cloned.
3. **Throughput**: A100 wall-clock per superbatch. Extrapolates the ~16-24 h budget that the Phase E plan assumes.
4. **Baseline**: produces a partial CReLU loss curve as a reference for the KAN variants in E.1.

**Cost**: 30 superbatches × 6104 batches × 16384 batch size ≈ 3 B sample-views. Estimated ~1 h on A100, ~$1.

## 1. Install Rust + clone repo

In [ ]:
%%bash
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version

In [ ]:
%%bash
set -e
rm -rf /content/bullet
cd /content
git clone https://github.com/y0sif/bullet.git
cd bullet
git log -1 --oneline
echo '---'
ls examples/phase_e0_smoke.rs
grep -A1 'phase_e0_smoke' crates/bullet_lib/Cargo.toml || echo 'WARN: phase_e0_smoke not in Cargo.toml'

## 2. Confirm A100 is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 3. Download January 2024 binpack (~7.7 GB compressed, ~25 GB decompressed)

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test80-2024-01-jan.binpack ]; then
    echo "Downloading test80-2024 January shard (~7.7 GB compressed)..."
    wget --progress=dot:giga -O test80-2024-01-jan.binpack.zst \
        "https://huggingface.co/datasets/linrock/test80-2024/resolve/main/test80-2024-01-jan-2tb7p.min-v2.v6.binpack.zst"
    echo "Decompressing..."
    zstd -d test80-2024-01-jan.binpack.zst -o test80-2024-01-jan.binpack --rm
fi

ls -lh test80-2024-01-jan.binpack
df -h /content

## 4. Build the smoke-test binary

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet
cargo build --release --example phase_e0_smoke 2>&1 | tail -15

## 5. Run training

In [ ]:
import subprocess, sys, os, shutil, time

os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

log_path = "/content/phase_e0_smoke_log.txt"
ckpt_dir = "/content/bullet/checkpoints"
if os.path.isdir(ckpt_dir):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

print(f"Logging to {log_path}")
start = time.time()

proc = subprocess.Popen(
    ["cargo", "run", "--release", "--example", "phase_e0_smoke"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd="/content/bullet", text=True, bufsize=1,
)
with open(log_path, "w") as log:
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        log.write(line)
proc.wait()

elapsed = time.time() - start
print(f"\nExit code: {proc.returncode}")
print(f"Wall-clock: {elapsed/60:.1f} min ({elapsed:.0f} s)")
if proc.returncode != 0:
    raise SystemExit("Training failed — inspect log before continuing.")

## 6. Parse loss curve + throughput report

In [ ]:
import re
import matplotlib.pyplot as plt

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

log_path = "/content/phase_e0_smoke_log.txt"
loss_records = []
time_records = []

with open(log_path) as f:
    for line in f:
        line = strip_ansi(line)
        m_loss = re.search(r'superbatch\s+(\d+)\s+\|.*?running loss\s+([\d.]+)', line)
        if m_loss:
            loss_records.append((int(m_loss.group(1)), float(m_loss.group(2))))
        m_time = re.search(r'superbatch\s+(\d+)\s+\|.*?(\d+\.\d+)s', line)
        if m_time:
            time_records.append((int(m_time.group(1)), float(m_time.group(2))))

if not loss_records:
    print("WARNING: no loss lines parsed — log format may have changed.")
else:
    first_sb, first_loss = loss_records[0]
    last_sb, last_loss = loss_records[-1]
    print(f"Loss: SB {first_sb}  {first_loss:.6f}  ->  SB {last_sb}  {last_loss:.6f}")
    drop = (first_loss - last_loss) / first_loss * 100
    print(f"Relative drop: {drop:.1f}%")

if time_records:
    secs = [t for _, t in time_records]
    mean_sb = sum(secs) / len(secs)
    print(f"\nPer-superbatch wall-clock (mean over {len(secs)} SB): {mean_sb:.1f} s")
    full_run_h = mean_sb * 800 / 3600
    print(f"Extrapolated 800-SB full run: {full_run_h:.1f} h")
    if full_run_h < 12 or full_run_h > 30:
        print("  >>> Outside the Phase E plan's 16-24 h budget. Re-evaluate before launching KAN runs.")
    else:
        print("  Within the Phase E plan's 16-24 h budget envelope.")
else:
    print("\nNo per-superbatch timing parsed. Use the cell-level wall-clock above as a coarser estimate.")

if loss_records:
    fig, ax = plt.subplots(figsize=(10, 5))
    xs = [sb for sb, _ in loss_records]
    ys = [l for _, l in loss_records]
    ax.plot(xs, ys, linewidth=2)
    ax.set_xlabel("Superbatch")
    ax.set_ylabel("Running loss")
    ax.set_title("Phase E.0 smoke test — CReLU baseline (Akimbo v1.0 recipe, 30 SB)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/content/phase_e0_smoke_loss.png", dpi=150)
    plt.show()

## 7. Save log + plot + quantised.bin to Drive

In [ ]:
import shutil, os, glob
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/kanue/phase_e0_smoke'
os.makedirs(dest, exist_ok=True)

for src in ['/content/phase_e0_smoke_log.txt', '/content/phase_e0_smoke_loss.png']:
    if os.path.exists(src):
        shutil.copy(src, dest)

for ckpt in sorted(glob.glob('/content/bullet/checkpoints/phase_e0_smoke-*')):
    name = os.path.basename(ckpt)
    out_dir = os.path.join(dest, name)
    os.makedirs(out_dir, exist_ok=True)
    for fname in ['quantised.bin', 'raw.bin', 'optim']:
        src = os.path.join(ckpt, fname)
        if os.path.exists(src) and os.path.isfile(src):
            shutil.copy(src, out_dir)
    print(f'Saved {name}')

print(f'\nDestination: {dest}')